### **1. Import, Load, Clean**

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Load dataset
PATH = "../data/raw/listings.csv"
df = pd.read_csv(PATH)

# Price cleaning
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Percentage cleaning
rates = ["host_response_rate", "host_acceptance_rate"]
for col in rates:
    df[col] = df[col].astype(str).str.replace("%", "").astype(float) / 100

# Remove rows where price is missing
df = df.dropna(subset=["price"]).copy()

In [3]:
cols_to_drop = [

    # identifiers / urls / metadata
    "id",
    "listing_url",
    "scrape_id",
    "last_scraped",
    "source",
    "picture_url",
    "host_id",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url",
    "calendar_updated",
    "calendar_last_scraped",

    # leakage
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d",

    # text fields (no NLP)
    "name",
    "description",
    "neighborhood_overview",
    "host_about",

    # redundant text versions
    "bathrooms_text",
    "host_name",
    "host_verifications",

    # review score redundancy
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",

    # review redundancy
    "number_of_reviews",

    # host listing redundancy
    "host_listings_count",
    "host_total_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",

    # availability redundancy
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_eoy",

    # review activity redundancy
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # derived night statistics
    "minimum_minimum_nights",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm",

    # categorical removal from EDA
    "host_since",
    "first_review",
    "last_review",
    "amenities",
    "license",
    "host_location",
    "host_neighbourhood",
    "neighbourhood",
    "neighbourhood_group_cleansed",
    "has_availability",
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",

    # weak categorical predictor
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate"
]

In [4]:
df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["price_bin"]
)

df_train = df_train.drop(columns=["price_bin"])
df_test = df_test.drop(columns=["price_bin"])

In [5]:
df_train["log_price"] = np.log1p(df_train["price"])
df_test["log_price"] = np.log1p(df_test["price"])

In [6]:
X_train = df_train.drop(["price", "log_price"], axis=1)
y_train = df_train["log_price"]

X_test = df_test.drop(["price", "log_price"], axis=1)
y_test = df_test["log_price"]

In [7]:
X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

In [8]:
print(X_train.shape)
print(X_test.shape)

(4520, 16)
(1131, 16)


In [9]:
X_train.columns

Index(['neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
       'minimum_nights', 'maximum_nights', 'availability_365',
       'review_scores_rating', 'instant_bookable',
       'calculated_host_listings_count', 'reviews_per_month'],
      dtype='str')

In [10]:
X_train.describe()

,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,availability_365,review_scores_rating,calculated_host_listings_count,reviews_per_month
count,4520.000000,4520.000000,4520.000000,4516.000000,4518.000000,4509.000000,4520.000000,4.520000e+03,4520.000000,4098.000000,4520.000000,4098.000000
mean,43.260544,-2.510491,4.043805,1.518379,1.992917,3.017964,3.103540,2.290641e+04,203.566593,4.749219,11.655088,1.549048
std,0.151415,0.435468,2.342981,1.019356,1.271072,2.362151,7.113805,1.487473e+06,119.513660,0.322975,26.532462,1.829270
min,42.487640,-3.378762,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000e+00,0.000000,1.000000,1.000000,0.010000
25%,43.257148,-2.925680,2.000000,1.000000,1.000000,1.000000,1.000000,1.800000e+02,86.000000,4.670000,1.000000,0.440000
50%,43.299497,-2.670280,4.000000,1.000000,2.000000,3.000000,2.000000,3.650000e+02,219.500000,4.830000,2.000000,1.000000
75%,43.323660,-1.985796,5.000000,2.000000,3.000000,4.000000,2.000000,1.125000e+03,322.000000,4.940000,8.000000,2.110000
max,43.441490,-1.757010,16.000000,24.000000,25.000000,48.000000,160.000000,1.000000e+08,365.000000,5.000000,149.000000,47.610000


### **2. Handling missing values**

In [11]:
missing = X_train.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0]

review_scores_rating    9.336283
reviews_per_month       9.336283
beds                    0.243363
bathrooms               0.088496
bedrooms                0.044248
dtype: float64

In [12]:
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include="number").columns

num_imputer = SimpleImputer(strategy="median")

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

In [13]:
cat_cols = X_train.select_dtypes(include="object").columns

cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_78805/1189760543.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


In [14]:
X_train.isna().sum().sum(), X_test.isna().sum().sum()

(np.int64(0), np.int64(0))

### **3. Transformations**

In [15]:
BILBAO_LAT = 43.2630
BILBAO_LON = -2.9350

def distance_to_bilbao(lat, lon):
    return np.sqrt((lat - BILBAO_LAT)**2 + (lon - BILBAO_LON)**2)

X_train["distance_to_bilbao"] = distance_to_bilbao(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_bilbao"] = distance_to_bilbao(
    X_test["latitude"], X_test["longitude"]
)

In [16]:
DONOSTIA_LAT = 43.3183
DONOSTIA_LON = -1.9812

def distance_to_donostia(lat, lon):
    return np.sqrt((lat - DONOSTIA_LAT)**2 + (lon - DONOSTIA_LON)**2)

X_train["distance_to_donostia"] = distance_to_donostia(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_donostia"] = distance_to_donostia(
    X_test["latitude"], X_test["longitude"]
)

In [17]:
VITORIA_LAT = 42.8467
VITORIA_LON = -2.6726

def distance_to_vitoria(lat, lon):
    return np.sqrt((lat - VITORIA_LAT)**2 + (lon - VITORIA_LON)**2)

X_train["distance_to_vitoria"] = distance_to_vitoria(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_vitoria"] = distance_to_vitoria(
    X_test["latitude"], X_test["longitude"]
)

In [18]:
COAST_LAT = 43.3623
COAST_LON = -3.0136

def distance_to_coast(lat, lon):
    return np.sqrt((lat - COAST_LAT)**2 + (lon - COAST_LON)**2)

X_train["distance_to_coast"] = distance_to_coast(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_coast"] = distance_to_coast(
    X_test["latitude"], X_test["longitude"]
)

In [19]:
X_train = X_train.drop(columns=["latitude", "longitude"])
X_test = X_test.drop(columns=["latitude", "longitude"])

In [20]:
clip_cols = [
    "beds",
    "minimum_nights",
    "maximum_nights"
]

for col in clip_cols:
    
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

In [21]:
skewed = [
    "minimum_nights",
    "maximum_nights",
    "reviews_per_month",
    "calculated_host_listings_count"
]

for col in skewed:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

In [22]:
cat_cols = X_train.select_dtypes(include="object").columns
cat_cols

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_78805/1847301736.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


Index(['neighbourhood_cleansed', 'property_type', 'room_type',
       'instant_bookable'],
      dtype='str')

In [23]:
X_train["instant_bookable"] = X_train["instant_bookable"].map({"t": 1, "f": 0})
X_test["instant_bookable"] = X_test["instant_bookable"].map({"t": 1, "f": 0})

In [24]:
TOP_K = 10

top_properties = (
    X_train["property_type"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

X_train["property_type_clean"] = X_train["property_type"].where(
    X_train["property_type"].isin(top_properties),
    "Other"
)

X_test["property_type_clean"] = X_test["property_type"].where(
    X_test["property_type"].isin(top_properties),
    "Other"
)

X_train = X_train.drop(columns=["property_type"])
X_test = X_test.drop(columns=["property_type"])

In [25]:
TOP_K = 10

top_neighbourhoods = (
    X_train["neighbourhood_cleansed"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

X_train["neighbourhood_cleansed_clean"] = X_train["neighbourhood_cleansed"].where(
    X_train["neighbourhood_cleansed"].isin(top_neighbourhoods),
    "Other"
)

X_test["neighbourhood_cleansed_clean"] = X_test["neighbourhood_cleansed"].where(
    X_test["neighbourhood_cleansed"].isin(top_neighbourhoods),
    "Other"
)

X_train = X_train.drop(columns=["neighbourhood_cleansed"])
X_test = X_test.drop(columns=["neighbourhood_cleansed"])

In [27]:
categorical_cols = [
    "property_type_clean",
    "room_type",
    "neighbourhood_cleansed_clean"
]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

### **4. Model**

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [29]:
from sklearn.linear_model import RidgeCV
import numpy as np

alphas = np.logspace(-3, 3, 50)

ridge_cv = RidgeCV(
    alphas=alphas,
    scoring="neg_root_mean_squared_error",
    cv=5
)

ridge_cv.fit(X_train_scaled, y_train)

print("Best alpha:", ridge_cv.alpha_)

Best alpha: 25.595479226995334


In [30]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=ridge_cv.alpha_)

ridge.fit(X_train_scaled, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",np.float64(25.595479226995334)
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochasti

In [31]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    ridge,
    X_train_scaled,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

rmse = -scores.mean()

print("CV RMSE:", rmse)

CV RMSE: 0.5285493914735335
